# Phase 6: Two-Tower Neural Retrieval (MovieLens 25M)

The optional deep-learning stretch from the project plan: a two-tower model (one neural network embedding users, one embedding items, scored by a dot product -- same scoring convention as ALS/BPR) trained on **this project's own processed MovieLens 25M split**, so it can be evaluated by the exact same `evaluate_model.py` every other model (popularity/itemknn/ALS/BPR/two-stage) already went through.

**Why this needs Colab**: everything else in this project runs locally on CPU in a reasonable time. Training a neural network this way benefits from a GPU for the in-batch softmax loss below (a full B×B score matrix per batch) -- that's the one piece of this project that's GPU-shaped, per the original plan.

**What you need to upload** (from your local `data/processed/movielens25m/` folder, generated back in Phase 0):
- `loo_train.parquet`
- `id_maps.json`

Use the file icon in Colab's left sidebar to upload them (they'll land in `/content/`), or mount Google Drive if you'd rather keep them there permanently. **Before running anything: Runtime → Change runtime type → GPU.**

**What this notebook does NOT do**: it doesn't run any evaluation itself. At the end, it exports two `.npy` embedding files for you to download and bring back to your local machine -- `scripts/build_two_tower_model.py` wraps them into a picklable model, and you evaluate it exactly like every other model in this project. That's deliberate: the whole point of Phase 6 is a *fair* comparison against Phases 2-3's numbers, which means reusing the identical evaluation code, not a separate Colab-side implementation of the same metrics.

In [ ]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if device.type != "cuda":
    print("WARNING: no GPU detected -- Runtime > Change runtime type > GPU, then re-run this cell.")

In [ ]:
# Load this project's own processed split -- same train data every other model fit on.
train = pd.read_parquet("loo_train.parquet")
id_maps = json.load(open("id_maps.json"))
n_users, n_items = id_maps["n_users"], id_maps["n_items"]
print(f"n_users={n_users:,} n_items={n_items:,} train_rows={len(train):,}")

user_ids_np = train["user_idx"].to_numpy()
item_ids_np = train["item_idx"].to_numpy()

# Per-item training frequency, used for the logQ correction below -- items are drawn
# into a batch proportional to how often they appear in train, so this is the
# "sampling probability" the correction needs.
item_counts = np.zeros(n_items, dtype=np.float64)
np.add.at(item_counts, item_ids_np, 1.0)
item_freq_np = (item_counts / item_counts.sum()).astype(np.float32)

## Architecture and training objective

**Two towers**: an embedding lookup for the user's ID, then a small MLP (`Linear -> ReLU -> Linear`), and the same shape of network for items. The MLP is the one thing ALS structurally can't do -- ALS's user/item vectors interact *only* through a plain dot product with nothing learned beyond that; here, each tower first passes its raw ID embedding through a nonlinear transformation before the dot product happens. If nonlinearity is going to earn its complexity over ALS's linear factorization, this is where it would show up.

Embedding dimension is set to 64, matching ALS's `factors=64` -- same representational capacity, so any accuracy difference is attributable to the architecture/training objective, not just "more numbers per user."

**Training loss: in-batch sampled softmax.** For a batch of `B` real (user, item) positive pairs, compute every user's score against every item *in that same batch* -- a `B x B` matrix. Each user's true positive is the diagonal entry; every other item in the batch (someone else's positive) stands in as a negative for this user, cheaply, with no separate negative-sampling step (unlike BPR's explicit triple sampling). This is the standard way large-scale two-tower retrieval models are trained in practice.

**Embeddings are L2-normalized before scoring, and logits are divided by a temperature.** This isn't a stylistic choice -- a first version of this notebook *without* normalization was run for real, and it collapsed: random item pairs came out with mean cosine similarity 0.997 (should be near 0), with the network differentiating items almost entirely by embedding *magnitude* instead of *direction*. Since a dot product with unnormalized vectors lets the loss be reduced just by growing/shrinking norms, the network found that cheap shortcut instead of learning real directional structure -- the resulting model's accuracy came in *below plain popularity*, and `intra_list_similarity` hit ~0.9997 (every recommended item nearly identical to every other). Normalizing both towers' outputs to unit length removes magnitude as a free parameter entirely, forcing direction to carry all the signal -- the standard fix used by real-world two-tower retrieval systems. `TEMPERATURE` (0.05 by default) rescales the now-bounded ([-1, 1]) cosine similarities back up before the softmax, since cosine similarities alone are too close together for cross-entropy to produce a useful gradient.

**Attempt 2's fix worked (collapse resolved, accuracy nearly doubled) but still landed below plain popularity** -- see LEARNINGS.md. Three further, specifically-targeted changes for attempt 3, not just "turn more dials":

1. **Bigger batches** (4096 -> 16384): more, and harder, in-batch negatives per training step -- a well-documented lever for contrastive/softmax representation quality, and cheap to try since nothing about the architecture changes.
2. **logQ correction** (the limitation flagged, but deliberately left unfixed, in attempts 1-2): in-batch negatives are sampled proportional to how often an item appears in training data, so a popular item is unfairly penalized just for showing up as *someone else's* positive more often -- not because the model scored it wrong. The standard correction (Yi et al., 2019, *"Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations"*): subtract `log(Q(i))` -- the item's empirical training frequency -- from every column's logits before the softmax, so the loss stops conflating "how often is this item a negative" with "how wrong was the model to score it this way."
3. **More epochs** (20 -> 40): bigger batches mean fewer gradient steps per epoch (same data, less frequent updates), so epochs are raised to compensate for that, not just to throw more compute at the problem blindly.

In [ ]:
EMBEDDING_DIM = 64
BATCH_SIZE = 16384
N_EPOCHS = 40
LEARNING_RATE = 1e-3
TEMPERATURE = 0.05

item_freq = torch.from_numpy(item_freq_np).to(device)


class Tower(nn.Module):
    def __init__(self, n_ids: int, embedding_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(n_ids, embedding_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim),
        )

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.embedding(ids))


class TwoTowerModel(nn.Module):
    def __init__(self, n_users: int, n_items: int, embedding_dim: int):
        super().__init__()
        self.user_tower = Tower(n_users, embedding_dim)
        self.item_tower = Tower(n_items, embedding_dim)

    def forward(self, user_ids: torch.Tensor, item_ids: torch.Tensor):
        return self.user_tower(user_ids), self.item_tower(item_ids)


model = TwoTowerModel(n_users, n_items, EMBEDDING_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

dataset = TensorDataset(torch.from_numpy(user_ids_np).long(), torch.from_numpy(item_ids_np).long())
# drop_last=True so every batch is exactly BATCH_SIZE -- the in-batch softmax needs a
# square (B, B) score matrix with a clean diagonal-as-positive-label
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"{len(loader):,} batches/epoch")

In [ ]:
for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0.0
    for user_batch, item_batch in loader:
        user_batch, item_batch = user_batch.to(device), item_batch.to(device)
        u, v = model(user_batch, item_batch)
        u = torch.nn.functional.normalize(u, dim=-1)
        v = torch.nn.functional.normalize(v, dim=-1)
        logits = (u @ v.T) / TEMPERATURE  # (B, B) -- row i's diagonal is the true positive

        # logQ correction: subtract each column item's own training-set log-frequency,
        # so a popular item isn't penalized just for appearing often as a negative
        log_q = torch.log(item_freq[item_batch] + 1e-10)  # (B,)
        logits = logits - log_q.unsqueeze(0)

        labels = torch.arange(len(user_batch), device=device)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"epoch {epoch + 1}/{N_EPOCHS}: mean batch loss = {total_loss / len(loader):.4f}")

## Export embeddings

Run every user ID and every item ID through its tower once (no gradient needed) to get the final, static `(n_users, 64)` / `(n_items, 64)` embedding tables -- these two arrays are the entire trained model, same role as ALS's `user_factors`/`item_factors`.

**Exported in L2-normalized form** (unit length), matching how they were scored during training (cosine similarity / temperature, not a raw dot product) -- `TwoTowerRecommender`'s plain dot product downstream then reproduces the same cosine-similarity ranking the model actually learned, rather than reintroducing the magnitude degree of freedom the normalization was added to remove.

In [ ]:
model.eval()
with torch.no_grad():
    user_embeddings = torch.nn.functional.normalize(
        model.user_tower(torch.arange(n_users, device=device)), dim=-1
    ).cpu().numpy().astype(np.float32)
    item_embeddings = torch.nn.functional.normalize(
        model.item_tower(torch.arange(n_items, device=device)), dim=-1
    ).cpu().numpy().astype(np.float32)

print(f"user_embeddings: {user_embeddings.shape}, item_embeddings: {item_embeddings.shape}")
np.save("user_embeddings.npy", user_embeddings)
np.save("item_embeddings.npy", item_embeddings)

In [ ]:
# Downloads both files to your local machine (two browser downloads). If you mounted
# Drive instead, just copy them there (e.g. shutil.copy(...)) rather than using this.
from google.colab import files

files.download("user_embeddings.npy")
files.download("item_embeddings.npy")

## Next step (back on your local machine)

Put both downloaded files somewhere like `artifacts/colab/`, then:

```bash
uv run python scripts/build_two_tower_model.py \
  --user-embeddings artifacts/colab/user_embeddings.npy \
  --item-embeddings artifacts/colab/item_embeddings.npy \
  --data-config configs/data_movielens25m.yaml

uv run python scripts/evaluate_model.py \
  --data-config configs/data_movielens25m.yaml \
  --eval-config configs/eval_default.yaml \
  --model-artifact artifacts/models/two_tower.pkl --split val
```

That's the exact same evaluation path ALS/BPR/two-stage went through -- see the README's Phase 6 RUNBOOK for the full writeup plan.